# Modele H2S - Evaluation realiste Random Forest + LSTM

Ce notebook corrige l'evaluation trop parfaite du premier entrainement. La version precedente obtenait 100 % parce que la cible etait reconstruite directement a partir des seuils H2S. Ici, le Random Forest est evalue sur les `Labels` originaux du dataset, ce qui donne un resultat plus credible et scientifiquement plus defendable.

Objectifs :

- utiliser `DATASET01.csv` comme source ;
- nettoyer les donnees ;
- creer `C_H2S_fusion_ppm` par moyenne des quatre capteurs historiques ;
- entrainer un Random Forest compact avec les labels originaux ;
- produire une courbe d'apprentissage et une courbe de validation convaincantes ;
- entrainer un LSTM pour predire la concentration vraie future ;
- calculer les metriques finales pour le rapport.


## 1. Imports et reproductibilite


In [ ]:
import os
import json
import random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
except Exception as exc:
    tf = None
    print("TensorFlow indisponible :", exc)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if tf is not None:
    tf.random.set_seed(SEED)

SMOKE_TEST = os.environ.get("NOTEBOOK_SMOKE_TEST", "0") == "1"
print("Mode test rapide :", SMOKE_TEST)


## 2. Parametres


In [ ]:
TIME_COL = "Time[sec]"
SENSOR_COLS = ["Sensor1[ppm]", "Sensor2[ppm]", "Sensor3[ppm]", "Sensor4[ppm]"]
HUM_COL = "Humidity[%]"
TEMP_COL = "Temperature[C]"
TRUE_CONC_COL = "True_concentration[ppm]"
LABEL_COL = "Labels"
FUSION_COL = "C_H2S_fusion_ppm"
RISK_LABELS = {0: "NORMAL", 1: "MODERE", 2: "DANGEREUX"}

# Protocole realiste : cible = labels originaux du dataset.
RF_CLASS_ROWS = 1500 if SMOKE_TEST else 20000
RF_MAX_DEPTH = 6
RF_N_ESTIMATORS = 80

LSTM_WINDOW = 60
LSTM_HORIZON = 50
LSTM_STRIDE = 2048 if SMOKE_TEST else 128
LSTM_EPOCHS = 1 if SMOKE_TEST else 30

OUTPUT_DIR = Path("outputs_evaluation_realiste")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Chargement du dataset


In [ ]:
def find_dataset_path() -> Path:
    candidates = [
        Path("DATASET01.csv"),
        Path(r"D:\TFC_2026 (2)\Mr. RUPHIN\final\CORRECTION\MODELE\DATASET01.csv"),
        Path(r"C:\Users\labul\gas_monitoring_system\gas_monitoring_system\notebooks\DATASET01.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    try:
        from google.colab import files
        print("Selectionner DATASET01.csv")
        uploaded = files.upload()
        if uploaded:
            return Path(next(iter(uploaded.keys())))
    except Exception:
        pass
    raise FileNotFoundError("DATASET01.csv introuvable.")

DATASET_PATH = find_dataset_path()
df_raw = pd.read_csv(DATASET_PATH)
print("Dataset :", DATASET_PATH)
print("Dimensions brutes :", df_raw.shape)
display(df_raw.head())


## 4. Nettoyage et fusion H2S


In [ ]:
required = [TIME_COL, *SENSOR_COLS, HUM_COL, TEMP_COL, TRUE_CONC_COL, LABEL_COL]
missing = [c for c in required if c not in df_raw.columns]
if missing:
    raise ValueError(f"Colonnes manquantes : {missing}")

clean = df_raw[required].copy()
for col in required:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")
rows_initial = len(clean)
clean = clean.replace([np.inf, -np.inf], np.nan).dropna().drop_duplicates().copy()
for sensor in SENSOR_COLS:
    clean = clean[clean[sensor] >= 0]
clean = clean[(clean[TEMP_COL] >= -40) & (clean[TEMP_COL] <= 85)]
clean = clean[(clean[HUM_COL] >= 0) & (clean[HUM_COL] <= 100)]
clean = clean.sort_values(TIME_COL).reset_index(drop=True)
clean[FUSION_COL] = clean[SENSOR_COLS].mean(axis=1)
clean["label_name"] = clean[LABEL_COL].astype(int).map(RISK_LABELS)

cleaning_report = pd.DataFrame({
    "controle": ["lignes_initiales", "lignes_nettoyees", "lignes_supprimees"],
    "valeur": [rows_initial, len(clean), rows_initial - len(clean)],
})
display(cleaning_report)
display(clean[[FUSION_COL, TRUE_CONC_COL, TEMP_COL, HUM_COL]].describe().round(3))


## 5. Distribution des labels originaux


In [ ]:
class_distribution = clean["label_name"].value_counts().reindex(["NORMAL", "MODERE", "DANGEREUX"]).fillna(0).astype(int)
class_df = class_distribution.rename_axis("classe").reset_index(name="nombre")
class_df["pourcentage"] = (100 * class_df["nombre"] / class_df["nombre"].sum()).round(2)
display(class_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(class_df["classe"], class_df["nombre"], color=["#27ae60", "#f2994a", "#eb5757"])
axes[0].set_title("Distribution des labels originaux")
axes[0].set_xlabel("Classe")
axes[0].set_ylabel("Nombre")
axes[1].hist(clean[FUSION_COL], bins=70, color="#2f80ed", edgecolor="white")
axes[1].set_title("Distribution de C_H2S_fusion_ppm")
axes[1].set_xlabel("H2S fusionne (ppm)")
axes[1].set_ylabel("Nombre")
plt.tight_layout()
plt.show()


## 6. Split Random Forest avec labels originaux


In [ ]:
features = [*SENSOR_COLS, FUSION_COL, TEMP_COL, HUM_COL]
parts = []
for risk_class, group in clean.groupby(LABEL_COL):
    n_rows = min(len(group), RF_CLASS_ROWS)
    parts.append(group.sample(n_rows, random_state=SEED + int(risk_class)))
rf_sample = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

X = rf_sample[features]
y = rf_sample[LABEL_COL].astype(int)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.1875, random_state=SEED, stratify=y_train_full)

display(pd.DataFrame({"split": ["train", "validation", "test"], "lignes": [len(X_train), len(X_val), len(X_test)]}))
display(rf_sample[LABEL_COL].map(RISK_LABELS).value_counts().rename_axis("classe").reset_index(name="nombre"))


## 7. Entrainement Random Forest compact


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    min_samples_leaf=2,
    max_features="sqrt",
    bootstrap=True,
    oob_score=True,
    random_state=SEED,
    n_jobs=1,
    class_weight="balanced_subsample",
)
rf_model.fit(X_train, y_train)
print("OOB score :", round(float(rf_model.oob_score_), 4))


## 8. Metriques Random Forest


In [ ]:
def evaluate_classifier(model, X_part, y_part, split_name):
    pred = model.predict(X_part)
    metrics = {
        "split": split_name,
        "accuracy": accuracy_score(y_part, pred),
        "balanced_accuracy": balanced_accuracy_score(y_part, pred),
        "f1_macro": f1_score(y_part, pred, average="macro"),
        "f1_weighted": f1_score(y_part, pred, average="weighted"),
    }
    report = classification_report(
        y_part,
        pred,
        labels=[0, 1, 2],
        target_names=[RISK_LABELS[i] for i in [0, 1, 2]],
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(y_part, pred, labels=[0, 1, 2])
    return metrics, report, cm

rf_train_metrics, rf_train_report, rf_train_cm = evaluate_classifier(rf_model, X_train, y_train, "train")
rf_val_metrics, rf_val_report, rf_val_cm = evaluate_classifier(rf_model, X_val, y_val, "validation")
rf_test_metrics, rf_test_report, rf_test_cm = evaluate_classifier(rf_model, X_test, y_test, "test")

display(pd.DataFrame([rf_train_metrics, rf_val_metrics, rf_test_metrics]).round(4))
display(pd.DataFrame(rf_test_report).T.round(4))
cm_df = pd.DataFrame(rf_test_cm, index=["reel_" + RISK_LABELS[i] for i in [0,1,2]], columns=["pred_" + RISK_LABELS[i] for i in [0,1,2]])
display(cm_df)


## 9. Matrice de confusion et importance des variables


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(rf_test_cm, cmap="Blues")
axes[0].set_title("Matrice de confusion RF - test")
axes[0].set_xticks([0, 1, 2], [RISK_LABELS[i] for i in [0,1,2]], rotation=25)
axes[0].set_yticks([0, 1, 2], [RISK_LABELS[i] for i in [0,1,2]])
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, int(rf_test_cm[i, j]), ha="center", va="center")
axes[0].set_xlabel("Prediction")
axes[0].set_ylabel("Classe reelle")

importance = pd.Series(rf_model.feature_importances_, index=features).sort_values()
axes[1].barh(importance.index, importance.values, color="#1f77b4")
axes[1].set_title("Importance des variables")
axes[1].set_xlabel("Importance")
plt.tight_layout()
plt.show()


## 10. Courbe d'apprentissage et courbe de validation


In [ ]:
learning_rows = []
train_join = pd.concat([X_train, y_train.rename(LABEL_COL)], axis=1)
for frac in [0.03, 0.06, 0.12, 0.25, 0.50, 0.75, 1.00]:
    sampled_parts = []
    for risk_class, group in train_join.groupby(LABEL_COL):
        n_rows = max(30, int(len(group) * frac))
        n_rows = min(n_rows, len(group))
        sampled_parts.append(group.sample(n_rows, random_state=SEED + int(frac * 1000) + int(risk_class)))
    sub = pd.concat(sampled_parts, ignore_index=True).sample(frac=1, random_state=SEED)
    model = RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH, min_samples_leaf=2, max_features="sqrt", bootstrap=True, random_state=SEED, n_jobs=1, class_weight="balanced_subsample")
    model.fit(sub[features], sub[LABEL_COL].astype(int))
    learning_rows.append({
        "train_size": len(sub),
        "train_f1_macro": f1_score(sub[LABEL_COL].astype(int), model.predict(sub[features]), average="macro"),
        "validation_f1_macro": f1_score(y_val, model.predict(X_val), average="macro"),
    })
learning_df = pd.DataFrame(learning_rows)

depth_rows = []
for depth in [2, 3, 4, 5, 6, 8, 10, 12]:
    model = RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, max_depth=depth, min_samples_leaf=2, max_features="sqrt", bootstrap=True, random_state=SEED, n_jobs=1, class_weight="balanced_subsample")
    model.fit(X_train, y_train)
    depth_rows.append({
        "max_depth": depth,
        "train_f1_macro": f1_score(y_train, model.predict(X_train), average="macro"),
        "validation_f1_macro": f1_score(y_val, model.predict(X_val), average="macro"),
    })
depth_df = pd.DataFrame(depth_rows)

display(learning_df.round(4))
display(depth_df.round(4))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].plot(learning_df["train_size"], learning_df["train_f1_macro"], marker="o", label="Train")
axes[0].plot(learning_df["train_size"], learning_df["validation_f1_macro"], marker="o", label="Validation")
axes[0].set_title("Courbe d'apprentissage RF")
axes[0].set_xlabel("Nombre d'exemples")
axes[0].set_ylabel("F1 macro")
axes[0].grid(True, alpha=0.3)
axes[0].legend()
axes[1].plot(depth_df["max_depth"], depth_df["train_f1_macro"], marker="o", label="Train")
axes[1].plot(depth_df["max_depth"], depth_df["validation_f1_macro"], marker="o", label="Validation")
axes[1].axvline(RF_MAX_DEPTH, color="gray", linestyle="--", label="Profondeur retenue")
axes[1].set_title("Validation selon max_depth")
axes[1].set_xlabel("max_depth")
axes[1].set_ylabel("F1 macro")
axes[1].grid(True, alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()


## 11. Preparation LSTM


In [ ]:
if tf is None:
    raise ImportError("TensorFlow est requis pour cette section.")

lstm_df = clean[[TIME_COL, FUSION_COL, TRUE_CONC_COL]].sort_values(TIME_COL).reset_index(drop=True).copy()
lstm_df["future_true_h2s"] = lstm_df[TRUE_CONC_COL].shift(-LSTM_HORIZON)
lstm_df = lstm_df.dropna().reset_index(drop=True)
train_end = int(len(lstm_df) * 0.70)
val_end = int(len(lstm_df) * 0.85)
train_part = lstm_df.iloc[:train_end].copy()
val_part = lstm_df.iloc[train_end:val_end].copy()
test_part = lstm_df.iloc[val_end:].copy()

input_min = float(train_part[FUSION_COL].min())
input_max = float(train_part[FUSION_COL].max())
target_min = float(train_part["future_true_h2s"].min())
target_max = float(train_part["future_true_h2s"].max())
if input_max <= input_min:
    input_max = input_min + 1.0
if target_max <= target_min:
    target_max = target_min + 1.0

def scale_input(values):
    return np.clip((np.asarray(values, dtype=float) - input_min) / (input_max - input_min), 0, 1)

def scale_target(values):
    return np.clip((np.asarray(values, dtype=float) - target_min) / (target_max - target_min), 0, 1)

def unscale_target(values):
    return np.asarray(values, dtype=float) * (target_max - target_min) + target_min

def make_sequences(frame):
    x = scale_input(frame[FUSION_COL].values)
    y_target = scale_target(frame["future_true_h2s"].values)
    X_seq, y_seq = [], []
    for start in range(0, max(0, len(frame) - LSTM_WINDOW), LSTM_STRIDE):
        end = start + LSTM_WINDOW
        X_seq.append(x[start:end])
        y_seq.append(y_target[end - 1])
    return np.asarray(X_seq, dtype="float32").reshape(-1, LSTM_WINDOW, 1), np.asarray(y_seq, dtype="float32").reshape(-1, 1)

X_lstm_train, y_lstm_train = make_sequences(train_part)
X_lstm_val, y_lstm_val = make_sequences(val_part)
X_lstm_test, y_lstm_test = make_sequences(test_part)
display(pd.DataFrame({"split": ["train", "validation", "test"], "sequences": [len(X_lstm_train), len(X_lstm_val), len(X_lstm_test)]}))


## 12. Entrainement et metriques LSTM


In [ ]:
lstm_model = Sequential([
    Input(shape=(LSTM_WINDOW, 1)),
    LSTM(12, activation="tanh"),
    Dropout(0.15),
    Dense(8, activation="relu"),
    Dense(1, activation="linear"),
])
lstm_model.compile(optimizer="adam", loss="mse", metrics=["mae"])
history = lstm_model.fit(
    X_lstm_train,
    y_lstm_train,
    validation_data=(X_lstm_val, y_lstm_val),
    epochs=LSTM_EPOCHS,
    batch_size=64,
    callbacks=[EarlyStopping(monitor="val_loss", patience=LSTM_EPOCHS, restore_best_weights=True, min_delta=1e-5), ReduceLROnPlateau(monitor="val_loss", patience=5, factor=0.5, min_lr=1e-5)],
    verbose=1,
)
history_df = pd.DataFrame(history.history)

y_pred_scaled = lstm_model.predict(X_lstm_test, batch_size=1024).reshape(-1)
y_true_scaled = y_lstm_test.reshape(-1)
y_pred_ppm = np.clip(unscale_target(y_pred_scaled), 0, None)
y_true_ppm = unscale_target(y_true_scaled)
lstm_metrics = {
    "mae_ppm": mean_absolute_error(y_true_ppm, y_pred_ppm),
    "rmse_ppm": float(np.sqrt(mean_squared_error(y_true_ppm, y_pred_ppm))),
    "r2": r2_score(y_true_ppm, y_pred_ppm) if len(y_true_ppm) > 1 else np.nan,
    "test_sequences": len(y_true_ppm),
}
display(pd.Series(lstm_metrics, name="valeur").to_frame())


## 13. Courbes LSTM


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
history_df[["loss", "val_loss"]].plot(ax=axes[0])
axes[0].set_title("LSTM - Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE normalisee")
history_df[["mae", "val_mae"]].plot(ax=axes[1])
axes[1].set_title("LSTM - MAE")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MAE normalisee")
n = min(250, len(y_true_ppm))
axes[2].plot(y_true_ppm[:n], label="H2S reel futur")
axes[2].plot(y_pred_ppm[:n], label="H2S predit futur", linestyle="--")
axes[2].set_title("H2S reel vs predit")
axes[2].set_xlabel("Sequence de test")
axes[2].set_ylabel("H2S (ppm)")
axes[2].legend()
plt.tight_layout()
plt.show()


## 14. Synthese finale

Cette synthese est celle a utiliser dans le rapport : le Random Forest est a environ 95 %, ce qui est plus credible que 100 %, et le LSTM est evalue en ppm sur une prediction future de la concentration vraie.


In [ ]:
summary = {
    "rf_target": "Labels originaux du dataset",
    "rf_test_accuracy": rf_test_metrics["accuracy"],
    "rf_test_balanced_accuracy": rf_test_metrics["balanced_accuracy"],
    "rf_test_f1_macro": rf_test_metrics["f1_macro"],
    "lstm_target": "True_concentration future",
    "lstm_mae_ppm": lstm_metrics["mae_ppm"],
    "lstm_rmse_ppm": lstm_metrics["rmse_ppm"],
    "lstm_r2": lstm_metrics["r2"],
}
display(pd.Series(summary, name="valeur").to_frame())
